# CHIME spectral index (alpha) statistics

Mean / median of `alpha` (and `alpha_err`) for the CHIME baseband and CHIME/FRB Catalog 1 samples.

In [1]:
import numpy as np
import pandas as pd



In [2]:

FILES = {
    "baseband": "/home/thsu/FRB_population/CHIME_spectra/baseband_spectra/baseband_spectral_index_summary.csv",
    "chimecat1": "/home/thsu/FRB_population/CHIME_spectra/chimecat1_spectra/chimecat1_alpha.csv",
}

dfs = {}
for name, path in FILES.items():
    df = pd.read_csv(path)
    df["frb_name"] = df["frb_name"].astype(str).str.strip()
    df["alpha"] = pd.to_numeric(df["alpha"], errors="coerce")
    df["alpha_err"] = pd.to_numeric(df["alpha_err"], errors="coerce")
    ndup = df["frb_name"].duplicated().sum()
    print(f"{name}: {len(df)} rows, {df['frb_name'].nunique()} unique FRBs, {ndup} duplicated names")
    dfs[name] = df

# Match on frb_name (inner join). Keep only FRBs with a finite alpha in BOTH catalogs
# so the two samples are identical in N.
# If a name is duplicated within a catalog, the first row is kept (check the printout above).
cols = ["frb_name", "alpha", "alpha_err"]
bb = dfs["baseband"][cols].drop_duplicates("frb_name")
c1 = dfs["chimecat1"][cols].drop_duplicates("frb_name")

matched = bb.merge(c1, on="frb_name", how="inner", suffixes=("_baseband", "_chimecat1"))
print(f"in both catalogs: {len(matched)}")
print(f"only in baseband: {len(set(bb.frb_name) - set(c1.frb_name))}")
print(f"only in chimecat1: {len(set(c1.frb_name) - set(bb.frb_name))}")

matched = matched[np.isfinite(matched["alpha_baseband"]) & np.isfinite(matched["alpha_chimecat1"])]
print(f"matched with finite alpha in both: {len(matched)}")
matched.head()

baseband: 133 rows, 133 unique FRBs, 0 duplicated names
chimecat1: 140 rows, 140 unique FRBs, 0 duplicated names
in both catalogs: 133
only in baseband: 0
only in chimecat1: 7
matched with finite alpha in both: 133


,frb_name,alpha_baseband,alpha_err_baseband,alpha_chimecat1,alpha_err_chimecat1
0,FRB20181209A,-1.828273,0.019538,-4.339904,0.255184
1,FRB20181213A,0.387140,0.031078,-0.953620,0.164049
2,FRB20181214C,-0.024834,0.009567,-0.443038,0.120292
3,FRB20181215B,0.957896,0.018435,0.587028,0.127708
4,FRB20181219C,-0.142930,0.013492,-1.730437,0.184809


In [3]:
def summarize(alpha, alpha_err):
    a = alpha.to_numpy()
    out = {
        "N": len(a),
        "mean": np.mean(a),
        "median": np.median(a),
        "std": np.std(a, ddof=1) if len(a) > 1 else np.nan,
        "sem": np.std(a, ddof=1) / np.sqrt(len(a)) if len(a) > 1 else np.nan,
        "mean_alpha_err": np.nanmean(alpha_err),
        "median_alpha_err": np.nanmedian(alpha_err),
    }
    # inverse-variance weighted mean (rows with positive, finite errors only)
    ok = np.isfinite(alpha_err) & (alpha_err > 0)
    if ok.any():
        w = 1.0 / alpha_err[ok].to_numpy() ** 2
        out["N_weighted"] = int(ok.sum())
        out["weighted_mean"] = np.sum(w * alpha[ok].to_numpy()) / np.sum(w)
        out["weighted_mean_err"] = 1.0 / np.sqrt(np.sum(w))
    return out

summary = pd.DataFrame({
    name: summarize(matched[f"alpha_{name}"], matched[f"alpha_err_{name}"])
    for name in ["baseband", "chimecat1"]
}).T
summary

,N,mean,median,std,sem,mean_alpha_err,median_alpha_err,N_weighted,weighted_mean,weighted_mean_err
baseband,133.0,-0.249106,-0.090779,0.685432,0.059435,0.023432,0.015670,133.0,-0.167945,0.001105
chimecat1,133.0,-1.029997,-1.066482,1.631203,0.141443,0.211772,0.194309,133.0,-0.852585,0.011645


In [4]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for name in ["baseband", "chimecat1"]:
    a = matched[f"alpha_{name}"]
    axes[0].hist(a, bins=25, histtype="step", label=f"{name} (N={len(a)}, median={a.median():.2f})")
axes[0].set_xlabel(r"spectral index $\alpha$")
axes[0].set_ylabel("count")
axes[0].legend()

axes[1].errorbar(matched["alpha_chimecat1"], matched["alpha_baseband"],
                 xerr=matched["alpha_err_chimecat1"], yerr=matched["alpha_err_baseband"],
                 fmt="o", ms=3, alpha=0.5, lw=0.5)
lim = np.array(axes[1].get_xlim())
axes[1].plot(lim, lim, "k--", lw=1)
axes[1].set_xlabel(r"$\alpha$ (CHIME Cat 1)")
axes[1].set_ylabel(r"$\alpha$ (baseband)")

plt.tight_layout()
plt.show()

/tmp/ipykernel_293010/3300091389.py:21: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
